# Day 54 — Custom Layers, Custom Models & Automatic Differentiation
- Create custom Keras layers by subclassing `keras.layers.Layer`
- Understand `build()` vs `call()`
- Work with trainable and non-trainable weights
- Create custom models by subclassing `keras.Model`
- Build losses and metrics that use model internals
- Understand TensorFlow automatic differentiation
- Use `tf.GradientTape`
- Manually calculate gradients and update model weights
- Connect the complete training pipeline: forward pass → loss → gradients → optimizer


## 1. Imports and setup

We will use TensorFlow/Keras throughout this notebook.

> If TensorFlow is not installed in your environment, install the version appropriate for your existing `tf_env` environment rather than changing environments unnecessarily.


In [29]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow version:", tf.__version__)
print("Keras version:", keras.__version__)


TensorFlow version: 2.21.0
Keras version: 3.15.1


## 2. Why custom layers?

Keras already provides many layers:

```python
layers.Dense(...)
layers.Dropout(...)
layers.BatchNormalization(...)
```

But sometimes a project requires a mathematical operation that is not directly available as a standard layer.

A custom layer lets us package that operation into a reusable Keras component.

The basic pattern is:

```python
class MyLayer(keras.layers.Layer):
    def __init__(self, ...):
        super().__init__()
        ...

    def build(self, input_shape):
        ...

    def call(self, inputs):
        ...
```

### Important distinction

- `__init__()` → configuration
- `build()` → create weights once the input shape is known
- `call()` → define the forward computation


In [30]:
class MyDense(keras.layers.Layer):
    def __init__(self, units, activation=None):
        super().__init__()
        self.units = units
        self.activation = keras.activations.get(activation)

    def build(self, input_shape):
        input_dim = input_shape[-1]

        self.kernel = self.add_weight(
            name="kernel",
            shape=(input_dim, self.units),
            initializer="glorot_uniform",
            trainable=True
        )

        self.bias = self.add_weight(
            name="bias",
            shape=(self.units,),
            initializer="zeros",
            trainable=True
        )

    def call(self, inputs):
        outputs = tf.matmul(inputs, self.kernel) + self.bias

        if self.activation is not None:
            outputs = self.activation(outputs)

        return outputs


custom_dense = MyDense(3, activation="relu")

X_demo = tf.random.normal((4, 5))
Y_demo = custom_dense(X_demo)

print("Output shape:", Y_demo.shape)
print("Trainable variables:", [v.name for v in custom_dense.trainable_variables])


Output shape: (4, 3)
Trainable variables: ['kernel', 'bias']


## 3. `build()` vs `call()`

A common question is:

**Why not create the kernel in `__init__()`?**

Because in many cases we do not yet know the number of input features.

For example:

```text
input shape = (batch_size, 5)
```

means the Dense-style kernel needs shape:

```text
(5, units)
```

Keras calls `build()` when the input shape becomes known.

This makes custom layers flexible and reusable.


In [31]:
print("Kernel shape:", custom_dense.kernel.shape)
print("Bias shape:", custom_dense.bias.shape)
print("Number of trainable variables:", len(custom_dense.trainable_variables))


Kernel shape: (5, 3)
Bias shape: (3,)
Number of trainable variables: 2


## 4. Trainable vs non-trainable weights

Keras automatically tracks weights created with:

```python
self.add_weight(...)
```

A trainable weight is updated by the optimizer.

A non-trainable weight is tracked by the layer but is not updated by gradient descent.

This is useful for quantities such as running statistics or counters.


In [15]:
class LayerWithCounter(keras.layers.Layer):
    def __init__(self):
        super().__init__()

        self.counter = self.add_weight(
            name="counter",
            shape=(),
            initializer="zeros",
            trainable=False
        )

    def call(self, inputs):
        self.counter.assign_add(1.0)
        return inputs


counter_layer = LayerWithCounter()

_ = counter_layer(tf.ones((2, 3)))
_ = counter_layer(tf.ones((2, 3)))
_ = counter_layer(tf.ones((2, 3)))
_ = counter_layer(tf.ones((2, 3)))

print("Counter:", counter_layer.counter.numpy())
print("Trainable weights:", counter_layer.trainable_weights)
print("Non-trainable weights:", counter_layer.non_trainable_weights)


Counter: 4.0
Trainable weights: []
Non-trainable weights: [<Variable path=layer_with_counter_1/counter, shape=(), dtype=float32, value=4.0>]


## 5. A custom layer with a regularization loss

A custom layer can add a loss during the forward pass.

This is especially useful when the loss depends on the layer's internal weights or activations.

Example: L2-style regularization on a kernel.


In [33]:
class RegularizedDense(keras.layers.Layer):
    def __init__(self, units, l2_strength=1e-4, activation=None):
        super().__init__()
        self.units = units
        self.l2_strength = l2_strength
        self.activation = keras.activations.get(activation)

    def build(self, input_shape):
        input_dim = input_shape[-1]

        self.kernel = self.add_weight(
            name="kernel",
            shape=(input_dim, self.units),
            initializer="glorot_uniform"
        )

        self.bias = self.add_weight(
            name="bias",
            shape=(self.units,),
            initializer="zeros"
        )

    def call(self, inputs):
        self.add_loss(self.l2_strength * tf.reduce_sum(tf.square(self.kernel)))

        outputs = tf.matmul(inputs, self.kernel) + self.bias

        if self.activation:
            outputs = self.activation(outputs)

        return outputs


reg_layer = RegularizedDense(4, l2_strength=1e-3, activation="relu")
_ = reg_layer(tf.random.normal((8, 5)))

print("Layer losses:", reg_layer.losses)


Layer losses: [<tf.Tensor: shape=(), dtype=float32, numpy=0.004223231691867113>]


## 6. Custom models with the Model subclassing API

For more complicated architectures, we can subclass `keras.Model`.

A model usually contains:

1. Layers defined in `__init__()`
2. Forward-pass logic in `call()`

Example:

```python
class MyModel(keras.Model):
    def __init__(self):
        super().__init__()
        ...

    def call(self, inputs):
        ...
```


In [34]:
class CustomMLP(keras.Model):
    def __init__(self, hidden_units=30, output_units=1):
        super().__init__()

        self.hidden = layers.Dense(hidden_units, activation="relu")
        self.output_layer = layers.Dense(output_units)

    def call(self, inputs):
        x = self.hidden(inputs)
        return self.output_layer(x)


model = CustomMLP()

X = tf.random.normal((10, 5))
y_pred = model(X)

print("Prediction shape:", y_pred.shape)
print("Trainable variables:", len(model.trainable_variables))


Prediction shape: (10, 1)
Trainable variables: 4


## 7. Model internals

One reason to subclass models is that you can explicitly expose intermediate computations.

For example, we can return both:

- the final prediction
- an internal representation

This is useful for debugging, visualization, custom losses, and explainability.


In [35]:
class InspectableMLP(keras.Model):
    def __init__(self, hidden_units=16):
        super().__init__()
        self.hidden = layers.Dense(hidden_units, activation="relu")
        self.output_layer = layers.Dense(1)

    def call(self, inputs, return_hidden=False):
        hidden = self.hidden(inputs)
        output = self.output_layer(hidden)

        if return_hidden:
            return output, hidden

        return output


inspectable_model = InspectableMLP()

predictions, hidden_features = inspectable_model(
    tf.random.normal((5, 4)),
    return_hidden=True
)

print("Predictions:", predictions.shape)
print("Hidden representation:", hidden_features.shape)


Predictions: (5, 1)
Hidden representation: (5, 16)


## 8. Custom loss based on model internals

A standard loss normally receives:

```python
loss(y_true, y_pred)
```

But sometimes we want a loss that also considers an internal representation.

For example:

```text
total loss =
prediction loss
+
small penalty on hidden activations
```

This can encourage the model to keep internal activations small.


In [36]:
def internal_activity_loss(y_true, y_pred, hidden):
    prediction_loss = tf.reduce_mean(tf.square(y_true - y_pred))
    activity_penalty = 1e-3 * tf.reduce_mean(tf.square(hidden))

    return prediction_loss + activity_penalty


X = tf.random.normal((20, 4))
y = tf.random.normal((20, 1))

pred, hidden = inspectable_model(X, return_hidden=True)

loss_value = internal_activity_loss(y, pred, hidden)

print("Custom loss:", loss_value.numpy())


Custom loss: 1.4374466


## 9. Custom metrics based on model internals

Metrics can also use internal information.

For example, suppose we want to track the average magnitude of hidden activations.

A reusable Keras metric can subclass `keras.metrics.Metric`.


In [37]:
class MeanActivationMagnitude(keras.metrics.Metric):
    def __init__(self, name="mean_activation_magnitude", **kwargs):
        super().__init__(name=name, **kwargs)

        self.total = self.add_weight(
            name="total",
            initializer="zeros"
        )
        self.count = self.add_weight(
            name="count",
            initializer="zeros"
        )

    def update_state(self, activations):
        value = tf.reduce_mean(tf.abs(activations))
        self.total.assign_add(value)
        self.count.assign_add(1.0)

    def result(self):
        return self.total / self.count

    def reset_state(self):
        self.total.assign(0.0)
        self.count.assign(0.0)


metric = MeanActivationMagnitude()

metric.update_state(hidden_features)
metric.update_state(hidden_features * 2)

print("Metric:", metric.result().numpy())

metric.reset_state()
print("After reset:", metric.result().numpy())


Metric: 0.36501676
After reset: nan


# 10. Automatic Differentiation — Autodiff

Automatic differentiation is one of the most important TensorFlow mechanisms.

Instead of manually deriving:

```text
dLoss / dWeight
```

TensorFlow records operations and automatically calculates gradients.

The main API is:

```python
with tf.GradientTape() as tape:
    ...

gradients = tape.gradient(loss, variables)
```

Think of `GradientTape` as a mathematical recorder.

```text
forward computation
       ↓
GradientTape records operations
       ↓
calculate loss
       ↓
backpropagation
       ↓
gradients
```


In [38]:
x = tf.Variable(3.0)

with tf.GradientTape() as tape:
    y = x ** 2

gradient = tape.gradient(y, x)

print("x:", x.numpy())
print("y:", y.numpy())
print("dy/dx:", gradient.numpy())

# Mathematically:
# y = x²
# dy/dx = 2x
# At x = 3, gradient = 6


x: 3.0
y: 9.0
dy/dx: 6.0


## 11. Multiple variables

GradientTape can calculate gradients with respect to multiple variables.

For:

```text
y = 3x² + 2z
```

we expect:

```text
dy/dx = 6x
dy/dz = 2
```


In [39]:
x = tf.Variable(2.0)
z = tf.Variable(5.0)

with tf.GradientTape() as tape:
    y = 3 * x**2 + 2 * z

gradients = tape.gradient(y, [x, z])

print("dy/dx:", gradients[0].numpy())
print("dy/dz:", gradients[1].numpy())


dy/dx: 12.0
dy/dz: 2.0


## 12. GradientTape with a neural network

Now we connect autodiff to a real model.

The process is:

1. Run the model
2. Calculate loss
3. Ask GradientTape for gradients
4. Apply gradients using an optimizer

This is essentially the core of neural-network training.


In [40]:
tf.random.set_seed(42)

X_train = tf.random.normal((100, 3))
y_train = 4 * X_train[:, :1] - 2 * X_train[:, 1:2] + 0.5 * X_train[:, 2:3]

model = keras.Sequential([
    layers.Dense(16, activation="relu"),
    layers.Dense(1)
])

optimizer = keras.optimizers.Adam(learning_rate=0.01)

for step in range(10):
    with tf.GradientTape() as tape:
        predictions = model(X_train)
        loss = tf.reduce_mean(tf.square(y_train - predictions))

    gradients = tape.gradient(loss, model.trainable_variables)

    optimizer.apply_gradients(
        zip(gradients, model.trainable_variables)
    )

    if step % 2 == 0:
        print(f"Step {step:2d} | Loss: {loss.numpy():.6f}")


Step  0 | Loss: 14.115593
Step  2 | Loss: 13.351609
Step  4 | Loss: 12.586705
Step  6 | Loss: 11.816765
Step  8 | Loss: 11.041945


## 13. Inspecting gradients

It is often useful to inspect gradients directly.

This helps diagnose:

- vanishing gradients
- exploding gradients
- incorrect loss functions
- frozen variables
- broken custom layers


In [41]:
with tf.GradientTape() as tape:
    predictions = model(X_train)
    loss = tf.reduce_mean(tf.square(y_train - predictions))

gradients = tape.gradient(loss, model.trainable_variables)

for variable, gradient in zip(model.trainable_variables, gradients):
    print(
        variable.name,
        "shape =", variable.shape,
        "gradient mean =",
        tf.reduce_mean(gradient).numpy()
    )


kernel shape = (3, 16) gradient mean = 0.16126782
bias shape = (16,) gradient mean = -0.22826043
kernel shape = (16, 1) gradient mean = 0.123504825
bias shape = (1,) gradient mean = -0.3288023


## 14. Persistent GradientTape

Normally a tape can be used once.

If we need multiple gradient calculations from the same recorded computation, use:

```python
tf.GradientTape(persistent=True)
```

Remember to delete the persistent tape when finished.


In [16]:
# 


x = tf.Variable(3.0)

with tf.GradientTape(persistent=True) as tape:
    y = x ** 3
    first_derivative = tape.gradient(y, x)
    second_derivative = tape.gradient(first_derivative, x)

print("First derivative:", first_derivative.numpy())
print("Second derivative:", second_derivative.numpy())

del tape

First derivative: 27.0
Second derivative: 18.0


## 15. Watching tensors manually

GradientTape normally watches trainable `tf.Variable` objects automatically.

For a normal tensor, explicitly request watching:

```python
tape.watch(x)
```


In [17]:
x = tf.constant(3.0)

with tf.GradientTape() as tape:
    tape.watch(x)
    y = x ** 2

gradient = tape.gradient(y, x)

print("Gradient:", gradient.numpy())


Gradient: 6.0


# 16. Complete custom training loop

Now we combine the ideas.

We will write a reusable training step using:

- model
- loss
- `GradientTape`
- gradients
- optimizer


In [60]:
tf.random.set_seed(123)

X_train = tf.random.normal((200, 4))
y_train = (
    2.5 * X_train[:, :1]
    - 1.5 * X_train[:, 1:2]
    + 0.7 * X_train[:, 2:3]
    + 0.2
)

custom_model = CustomMLP(hidden_units=20)
optimizer = keras.optimizers.Adam(learning_rate=0.01)


def train_step(model, X, y, optimizer):
    with tf.GradientTape() as tape:
        y_pred = model(X)
        loss = tf.reduce_mean(tf.square(y - y_pred))

    gradients = tape.gradient(
        loss,
        model.trainable_variables
    )

    optimizer.apply_gradients(
        zip(gradients, model.trainable_variables)
    )

    return loss


for epoch in range(20):
    loss = train_step(
        custom_model,
        X_train,
        y_train,
        optimizer
    )

    if epoch % 2 == 0:
        print(f"Epoch {epoch:2d} | Loss: {loss.numpy():.6f}")


Epoch  0 | Loss: 7.877244
Epoch  2 | Loss: 6.997110
Epoch  4 | Loss: 6.186002
Epoch  6 | Loss: 5.438681
Epoch  8 | Loss: 4.750113
Epoch 10 | Loss: 4.109822
Epoch 12 | Loss: 3.515164
Epoch 14 | Loss: 2.963403
Epoch 16 | Loss: 2.459268
Epoch 18 | Loss: 2.002077


## 17. Compare manual training with `model.fit()`

Keras normally hides much of the training loop:

```python
model.compile(...)
model.fit(...)
```

A custom training loop exposes the individual steps:

```text
model.fit()
   ↓
forward pass
   ↓
loss
   ↓
GradientTape
   ↓
gradients
   ↓
optimizer
   ↓
metrics
```

Use the high-level API when it is sufficient. Use custom training loops when you need specialized training behavior.
